In [ ]:
import gc
import itertools
import logging
import os
import tomllib
from pathlib import Path

import gensim
import h5py
import nltk
import numpy as np
import pandas as pd
import scipy
import tqdm
from annoy import AnnoyIndex
from gensim.corpora import Dictionary
from gensim.matutils import sparse2full
from gensim.models import Word2Vec

# from gensim.test.utils import datapath
from gensim.models.phrases import ENGLISH_CONNECTOR_WORDS, Phrases
from gensim.models.tfidfmodel import TfidfModel
from gensim.parsing.preprocessing import remove_stopwords
from nltk import sent_tokenize
from nltk.corpus import stopwords


In [ ]:
nltk.download('punkt')
nltk.download('stopwords')

stopWords = set(stopwords.words('english'))
stopwords = nltk.corpus.stopwords.words('english')

logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

[nltk_data] Downloading package punkt to
[nltk_data]     /home/frap@ip.local/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/frap@ip.local/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/frap@ip.local/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
with open("./config.toml", "rb") as f:
    config = tomllib.load(f)

INPUT_FILE = Path(config["path"]["input"]) / "CN_abstracts.parquet"
OUTPUT = Path(config["path"]["output"])
OUTPUT.mkdir(parents=True, exist_ok=True)
INTERMED = Path(config["path"]["intermed"])
INTERMED.mkdir(parents=True, exist_ok=True)

# hyps
bi_min_count = config["bigram"]["min_count"]
bi_threshold = config["bigram"]["threshold"]
vector_size = config["word2vec"]["vector_size"]
window = config["word2vec"]["window"]
w2v_min_count = config["word2vec"]["min_count"]
epochs = config["word2vec"]["epochs"]
trees = config["similarity"]["trees"]
search_k = config["similarity"]["search_k"]
thr = config["similarity"]["thr"]
upto = config["similarity"]["upto"]

WORKERS = int(max(os.cpu_count()/2, 1))

In [59]:
print('Nr of workers used = ', WORKERS)

Nr of workers used =  12


In [ ]:
# Preprocessing (sentence tokenizer + basik Gensim - lowercases, tokenizes, de-accents (optional))
def prepro(x):
    x = sent_tokenize(x)
    clean = []
    for j in x:
        clean.append(gensim.utils.simple_preprocess(j, min_len=1)) # min_len=1 -> keeps single letter token
    return clean

def bigrammer(x):
    sents = []
    for i in x:
        sents.append(bigram[i])
    return sents

def tfidfer(para):
    return model_tfidf[docs_dict.doc2bow(itertools.chain(*para))]

def preprocess_1 (data):
    data['appln_abstract_prepro'] = list(map(prepro, data['abstract']))
    return data

def preprocess_2 (data):
    data['appln_abstract_prepro_bi'] = list(map(bigrammer, data['appln_abstract_prepro']))
    return data

def preprocess_3 (data):
    data['tfidf'] = list(map(tfidfer, data['appln_abstract_prepro_bi']))
    return data

In [ ]:
data = pd.read_parquet(INPUT_FILE, engine="pyarrow")
data = data[data['abstract'].notnull() & data['abstract_lg'].eq('en')]

In [ ]:
data = data.head(100) ### test, remove this line

In [44]:
print('Total appln to process = ', len(data))

Total appln to process =  5


In [46]:
data = preprocess_1(data) # add prepro

In [ ]:
# build bigrams
sentences = [sent for doc in data['appln_abstract_prepro'] for sent in doc if sent]
bigram = Phrases(sentences, min_count=bi_min_count, threshold=bi_threshold, connector_words=ENGLISH_CONNECTOR_WORDS)
bigram = gensim.models.phrases.Phraser(bigram)
bigram.save(INTERMED / 'bigram.m')

2026-11-03 17:18:28,754 : INFO : collecting all words and their counts
2026-11-03 17:18:28,755 : INFO : PROGRESS: at sentence #0, processed 0 words and 0 word types
2026-11-03 17:18:28,756 : INFO : collected 427 token types (unigram + bigrams) from a corpus of 313 words and 19 sentences
2026-11-03 17:18:28,757 : INFO : merged Phrases<427 vocab, min_count=5, threshold=1, max_vocab_size=40000000>
2026-11-03 17:18:28,757 : INFO : Phrases lifecycle event {'msg': 'built Phrases<427 vocab, min_count=5, threshold=1, max_vocab_size=40000000> in 0.00s', 'datetime': '2026-11-03T17:18:28.757882', 'gensim': '4.4.0', 'python': '3.12.16 (main, Oct 26 2026, 11:37:43) [GCC 15.3.0]', 'platform': 'Linux-6.8.0-138-generic-x86_64-with-glibc2.35', 'event': 'created'}
2026-11-03 17:18:28,758 : INFO : exporting phrases from Phrases<427 vocab, min_count=5, threshold=1, max_vocab_size=40000000>
2026-11-03 17:18:28,760 : INFO : FrozenPhrases lifecycle event {'msg': 'exported FrozenPhrases<0 phrases, min_count=5

In [48]:
data = preprocess_2(data) # add bigram

In [ ]:
# word to vec
sentences = [sent for doc in data['appln_abstract_prepro_bi'] for sent in doc if sent] # using bigram
model_with_phrases = Word2Vec(sentences=sentences, vector_size=vector_size, window=window,
                              min_count=w2v_min_count, epochs=epochs, workers=WORKERS)
model_with_phrases.save(INTERMED / 'w2v.m')

2026-11-03 17:18:33,109 : INFO : collecting all words and their counts
2026-11-03 17:18:33,111 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-11-03 17:18:33,112 : INFO : collected 192 word types from a corpus of 313 raw words and 19 sentences
2026-11-03 17:18:33,113 : INFO : Creating a fresh vocabulary
2026-11-03 17:18:33,114 : INFO : Word2Vec lifecycle event {'msg': 'effective_min_count=5 retains 8 unique words (4.17% of original 192, drops 184)', 'datetime': '2026-11-03T17:18:33.114260', 'gensim': '4.4.0', 'python': '3.12.16 (main, Oct 26 2026, 11:37:43) [GCC 15.3.0]', 'platform': 'Linux-6.8.0-138-generic-x86_64-with-glibc2.35', 'event': 'prepare_vocab'}
2026-11-03 17:18:33,115 : INFO : Word2Vec lifecycle event {'msg': 'effective_min_count=5 leaves 51 word corpus (16.29% of original 313, drops 262)', 'datetime': '2026-11-03T17:18:33.115441', 'gensim': '4.4.0', 'python': '3.12.16 (main, Oct 26 2026, 11:37:43) [GCC 15.3.0]', 'platform': 'Linux-6.8.0-138

In [ ]:
# Build Gensim Dictionary
voc = model_with_phrases.wv.index_to_key
docs_dict = Dictionary([voc])
docs_dict.compactify()
docs_dict.save(INTERMED / 'docs_dict_11_2.d')

2026-11-03 17:20:18,031 : INFO : adding document #0 to Dictionary<0 unique tokens: []>
2026-11-03 17:20:18,033 : INFO : built Dictionary<8 unique tokens: ['atm', 'device', 'digital', 'dispense', 'files']...> from 1 documents (total 8 corpus positions)
2026-11-03 17:20:18,034 : INFO : Dictionary lifecycle event {'msg': "built Dictionary<8 unique tokens: ['atm', 'device', 'digital', 'dispense', 'files']...> from 1 documents (total 8 corpus positions)", 'datetime': '2026-11-03T17:20:18.034007', 'gensim': '4.4.0', 'python': '3.12.16 (main, Oct 26 2026, 11:37:43) [GCC 15.3.0]', 'platform': 'Linux-6.8.0-138-generic-x86_64-with-glibc2.35', 'event': 'created'}
2026-11-03 17:20:18,035 : INFO : Dictionary lifecycle event {'fname_or_handle': 'docs_dict_11_2.d', 'separately': 'None', 'sep_limit': 10485760, 'ignore': frozenset(), 'datetime': '2026-11-03T17:20:18.035510', 'gensim': '4.4.0', 'python': '3.12.16 (main, Oct 26 2026, 11:37:43) [GCC 15.3.0]', 'platform': 'Linux-6.8.0-138-generic-x86_64-wi

In [ ]:
# tfidf at patent level
docs = [list(itertools.chain(*doc)) for doc in data['appln_abstract_prepro_bi']]
model_tfidf = TfidfModel((docs_dict.doc2bow(d) for d in docs), id2word=docs_dict)
model_tfidf.save(INTERMED / 'model_tfidf_11_2.m')

docs_vecs = (sparse2full(model_tfidf[docs_dict.doc2bow(d)], len(docs_dict)) for d in docs)

2026-11-03 17:21:47,353 : INFO : collecting document frequencies
2026-11-03 17:21:47,354 : INFO : PROGRESS: processing document #0
2026-11-03 17:21:47,355 : INFO : TfidfModel lifecycle event {'msg': 'calculated IDF weights for 5 documents and 8 features (12 matrix non-zeros)', 'datetime': '2026-11-03T17:21:47.355861', 'gensim': '4.4.0', 'python': '3.12.16 (main, Oct 26 2026, 11:37:43) [GCC 15.3.0]', 'platform': 'Linux-6.8.0-138-generic-x86_64-with-glibc2.35', 'event': 'initialize'}
2026-11-03 17:21:47,357 : INFO : TfidfModel lifecycle event {'fname_or_handle': 'model_tfidf_11_2.m', 'separately': 'None', 'sep_limit': 10485760, 'ignore': frozenset(), 'datetime': '2026-11-03T17:21:47.357167', 'gensim': '4.4.0', 'python': '3.12.16 (main, Oct 26 2026, 11:37:43) [GCC 15.3.0]', 'platform': 'Linux-6.8.0-138-generic-x86_64-with-glibc2.35', 'event': 'saving'}
2026-11-03 17:21:47,501 : INFO : saved model_tfidf_11_2.m


In [ ]:
data = preprocess_3(data)
data.to_parquet(INTERMED / "data.parquet", engine="pyarrow", index=False)

In [ ]:
# Selected Word-Embeddings
emb_vecs_selftrained = np.vstack([model_with_phrases.wv[docs_dict.get(i)] for i in range(len(docs_dict))]) # n*300
n_abstracts = len(data)
appln_ids = data['appln_id'].to_numpy()          # same order as docs / docs_vecs

with h5py.File(INTERMED / 'docvecs_23_4_test.h5', 'w') as h5f:     # 'w' = overwrite, safe to rerun
    dataset = h5f.create_dataset('weighted_tfidf_docvecs', (n_abstracts, vector_size), dtype='f4')
    h5f.create_dataset('appln_id', data=appln_ids)

    a = np.vstack([next(docs_vecs) for _ in range(n_abstracts)])
    dataset[:] = np.dot(a, emb_vecs_selftrained)

In [ ]:
h5f = h5py.File(INTERMED / 'docvecs_23_4_test.h5', 'r')
dataset = h5f['weighted_tfidf_docvecs']
ids = h5f['appln_id'][:] # get every element
row_of = {aid: i for i, aid in enumerate(ids)}      # appln_id -> row

def vec(appln_id):
    return dataset[row_of[appln_id]]

scipy.spatial.distance.cosine(vec(ids[1]), vec(ids[0]))   # or vec(<a real appln_id>)

/home/frap@ip.local/miniconda3/envs/patent_novelty/lib/python3.12/site-packages/scipy/spatial/distance.py:670: RuntimeWarning: invalid value encountered in scalar divide
  dist = 1.0 - uv / math.sqrt(uu * vv)


np.float32(nan)

In [ ]:
# build trees
t = AnnoyIndex(vector_size, 'dot')
t.on_disk_build(str(INTERMED / 'vecs_build.annoy'))
for row in tqdm.tqdm(range(len(dataset))):
    t.add_item(row, dataset[row])       # Annoy item = row number
t.build(trees)
t.unload()                              # release the file (needed on Windows before re-loading)

/tmp/ipykernel_365587/2192281853.py:2: FutureWarning: The default argument for metric will be removed in future version of Annoy. Please pass metric='angular' explicitly.
  t = AnnoyIndex(dataset.shape[1])
100%|██████████| 5/5 [00:00<00:00, 33.92it/s]


True

In [ ]:
tt = AnnoyIndex(vector_size, 'dot')
tt.load(str(INTERMED / 'vecs_build.annoy'), prefault=False)

simH5 = h5py.File(OUTPUT / 'similarities_23_2.h5', 'w')
dataset = simH5.create_dataset('sims_50',(10000000,3), maxshape=(None,3), dtype='f8', compression="gzip", )

In [ ]:
pbar = tqdm.tqdm(total=tt.get_n_items())

blocks = []
start = 0
end = 0
counter = 0
for i in range(tt.get_n_items()):
    j = 100
    tresh = 1.0
    while tresh >= thr:
        y = tt.get_nns_by_item(i, j, search_k=search_k, include_distances=True)
        tresh = y[1][-1]
        if len(y[0]) < j:
            break
        j += 50
        if j >= upto:
            break
    block = np.array(list(zip([i for _ in range(len(y[0]))],y[0],y[1])))
    end = end + len(block)
    blocks.append(block)
    counter += 1
    if counter == 1000:
        counter = 0
        blocks_np = np.vstack(blocks)
        if end >= dataset.shape[0]:
            dataset.resize((end+(end - dataset.shape[0]),3))
        dataset[start:end] = blocks_np
        blocks = []
        gc.collect()
        start = end
        pbar.update(1000)

if blocks:
    blocks_np = np.vstack(blocks)
    if end > dataset.shape[0]:
        dataset.resize((end, 3))
    dataset[start:end] = blocks_np
    pbar.update(counter)
    blocks = []

# trim the unused pre-allocated rows of zeros
dataset.resize((end, 3))

pbar.close()
simH5.close()

In [ ]:
def lookup(x):
    y = tt.get_nns_by_item(x, j, search_k=search_k, include_distances=True)
    return np.array(list(zip([x for _ in range(len(y[0]))],y[0],y[1])))

In [ ]:
lookup(0)
